In [17]:
# Install chromadb to store and search vector embeddings
!pip install -q chromadb
# Install the modern Google GenAI SDK
!pip install -q google-genai
# Import the client for Google Gemini
from google import genai
# Import chromadb for civic complaint vector storage
import chromadb
# Print setup confirmation to verify everything works
print("Setup ready")


Setup ready


In [2]:
#Compiled from official Rajasthan government pages; screenshots kept as evidence.

documents = [
    # 1. Garbage Dumping & Municipal Solid Waste (Source: jaipurmc.org & JMC Officer Directory PDF)
    "Issue: Garbage dumping and uncollected municipal solid waste (Keywords: kachra, kooda, solid waste, overflowing dustbin, street litter, safai, garbage pile). Authority: Nagar Nigam Jaipur (Jaipur Municipal Corporation) - Deputy Commissioner (Health) & Executive Engineer (Solid Waste Management). Submit via: JMC Call Centre / Public Grievance Helpline 0141-2747400 (08:00 AM to 10:00 PM, 7 days), 'Jaipur Samadhan' Android mobile app, or online at http://jaipurmc.org (About Us -> Online Services -> Register Your Complaint). Rule: Solid Waste Management Rules, 2026 & Rajasthan Municipalities Act, 2009 (Section 226). Escalate to: Commissioner JMC (0141-2740510, commissioner.jmc@rajasthan.gov.in) or JMC Control Room (0141-2742900). Source: http://jaipurmc.org/Presentation/PublicNotice/FAQs.aspx (Verified: 23-11-2026).",

    # 2. Construction & Demolition (C&D) Waste Dumping (Source: RSPCB CandDWaste.html & ComplaintPage.html)
    "Issue: Construction and demolition waste dumping or malba accumulation (Keywords: C&D waste, malba, construction debris, building waste, rubble, patthar, mitti). Authority: Rajasthan State Pollution Control Board (RSPCB). Submit via: Rajasthan Sampark Portal (https://sampark.rajasthan.gov.in) under Public Grievance with location, nature of pollution, and photos/videos. Rule: Construction and Demolition Waste Management Rules, 2016. Escalate to: Rajasthan Sampark Helpline 181 or Department of Public Grievances (0141-2922825, ds.rpg@rajasthan.gov.in). Source: https://environment.rajasthan.gov.in/content/environment/en/rajasthan-state-pollution-control-board/ComplaintPage.html (Verified: 23-11-2026).",

    # 3. Waste / Open Burning (Source: RSPCB ComplaintPage.html & sampark.rajasthan.gov.in)
    "Issue: Waste and open garbage burning (Keywords: kachra jalana, kooda jalana, patte jalana, dry leaves burning, plastic burning, garbage burning, smoke, dhuan, aag lagana). Authority: Rajasthan State Pollution Control Board (RSPCB). Submit via: Rajasthan Sampark Portal (https://sampark.rajasthan.gov.in) under Public Grievance with location, nature of pollution, and photos/videos. Rule: Solid Waste Management Rules, 2026 (Rule 5(1)(f) - no burning of waste on streets or open public spaces; Rule 17 - Environmental Compensation, polluter pays principle) & Air (Prevention and Control of Pollution) Act, 1981. Escalate to: Rajasthan Sampark Toll-Free Helpline 181 (0141-2922272, 2921646) or Department of Public Grievances (ds.rpg@rajasthan.gov.in). Source: https://environment.rajasthan.gov.in/content/environment/en/rajasthan-state-pollution-control-board/ComplaintPage.html (Verified: 23-11-2026).",

    # 4. Water Supply Leakage & Pipeline Burst (Source: PHED Control Room page & District Control Room PDF)
    "Issue: Drinking water supply leakage, pipeline burst, or water wastage (Keywords: paani leak, pipeline burst, broken water pipe, water wastage, nal ka paani behna, drinking water supply, jal). Authority: Public Health Engineering Department (PHED), Government of Rajasthan - Jaipur District Control Room. Submit via: Jaipur District Control Room Phone 0141-2706624 or PHED State Control Room 0141-2222585. Rule: not available. Escalate to: Chief Engineer (Urban & NRW), PHED Jaipur (0141-2222053, rj_cehq@nic.in) or Rajasthan Sampark Helpline 181. Source: https://sampark.rajasthan.gov.in (Verified: 24-11-2026).",

    # 5. Sewer Overflow & Choked Storm Drains (Source: jaipurmc.org & JMC Officer Directory PDF)
    "Issue: Sewer overflow, choked drains, and dirty water stagnation (Keywords: naali jaam, sewer overflow, gatar, drainage blockage, dirty water stagnation, sewage leak). Authority: Nagar Nigam Jaipur (Jaipur Municipal Corporation) - Superintending Engineer (Sewer Project) / Executive Engineer (Liquid Waste Management). Submit via: JMC Call Centre 0141-2747400 (08:00 AM to 10:00 PM), 'Jaipur Samadhan' app, or http://jaipurmc.org (Online Services -> Register Your Complaint). Rule: Rajasthan Municipalities Act, 2009 (Chapter XI - Drainage and Sewerage) & Water (Prevention and Control of Pollution) Act, 1974. Escalate to: JMC Control Room (0141-2742900) or Commissioner JMC (commissioner.jmc@rajasthan.gov.in). Source: http://jaipurmc.org/Presentation/Others/ConctactUs.aspx (Verified: 23-11-2026).",

    # 6. Tree Cutting & Forest Protection (Source: forest.rajasthan.gov.in & sampark.rajasthan.gov.in)
    "Issue: Illegal tree cutting, felling, or damage to forest trees (Keywords: ped katna, tree cutting, illegal tree felling, branch cutting, jungle, forest destruction). Authority: Forest Department, Government of Rajasthan. Submit via: Rajasthan Sampark Portal (https://rajasthan.gov.in -> Parivad Panjikaran) or Toll-Free Helpline 181 (0141-2922272, 2921646). Rule: The Rajasthan Forest Act, 1953 and Rajasthan Forest Policy 2023. Escalate to: Department of Public Grievances, Secretariat Jaipur (0141-2922825, 0141-2385077, ds.rpg@rajasthan.gov.in). Source: https://sampark.rajasthan.gov.in (Verified: 24-11-2026).",

    # 7. Biodiversity & Heritage Grove Destruction (Source: Rajasthan State Biodiversity Board Grievance & Functions pages)
    "Issue: Destruction of local biodiversity, heritage sites, sacred groves, or wildlife habitats (Keywords: biodiversity damage, sacred grove, heritage trees, flora fauna harm, wildlife disturbance). Authority: Rajasthan State Biodiversity Board, Jaipur (Member Secretary / Chief Manager Technical). Submit via: Phone 0141-2362188, 0141-2362088 or Email rajsbb@rajasthan.gov.in / rajasthansbb@gmail.com (Address: Arawali Bhawan, First Floor, Jhalana Institutional Area, Jaipur 302004). Rule: Biological Diversity Act, 2002 and Rajasthan Biological Diversity Rules, 2010. Escalate to: Rajasthan Sampark Portal (https://rajasthan.gov.in) or Helpline 181. Source: https://sampark.rajasthan.gov.in (Verified: 24-11-2026).",

    # 8. Noise & Air Pollution (Source: RSPCB ComplaintPage.html & sampark.rajasthan.gov.in)
    "Issue: Noise pollution and industrial or vehicular air pollution (Keywords: tez awaaz, shor, loudspeaker, DJ noise, late night music, generator noise, factory smoke, dhuan, air pollution, dust). Authority: Rajasthan State Pollution Control Board (RSPCB). Submit via: Rajasthan Sampark Portal (https://sampark.rajasthan.gov.in) under Public Grievance with location, nature of pollution, and photos/videos. Rule: Noise Pollution (Regulation and Control) Rules, 2000 (Rule 5) & Air (Prevention and Control of Pollution) Act, 1981. Escalate to: Rajasthan Sampark Toll-Free Helpline 181 (0141-2922272, 2921646) or Department of Public Grievances (ds.rpg@rajasthan.gov.in). Source: https://rajasthan.gov.in (Verified: 23-11-2026).",

    # 9. General Unresolved Civic Grievance Escalation (Source: sampark.rajasthan.gov.in/ContactUs)
    "Issue: General civic or environmental grievance escalation when no action is taken (Keywords: unresolved complaint, no action, sunwai nahi hui, escalation, shikayat, pending complaint, parivad panjikaran). Authority: Department of Public Grievances, Government of Rajasthan (Rajasthan Sampark). Submit via: Rajasthan Sampark Portal (https://rajasthan.gov.in -> Parivad Panjikaran), Toll-Free Helpline 181, Call Center (0141-2922272, 2921646), Tech Helpdesk (0141-2922293), or Email rajsampark@rajasthan.gov.in. Rule: not available. Escalate to: Director of Public Grievances & Ex-Officio Special Secretary - Room No. 7220, Food Building, 2nd Floor, Secretariat, Jaipur (0141-2922825, 0141-2385077, ds.rpg@rajasthan.gov.in). Source: https://rajasthan.gov.in/ContactUs (Verified: 24-11-2026)."
]

# Automatically generate matching IDs for all 9 documents
ids = [f"complaint_{i}" for i in range(1, len(documents) + 1)]

print(f"Number of verified documents: {len(documents)}")
print(f"Number of IDs: {len(ids)}")


Number of verified documents: 9
Number of IDs: 9


In [3]:
import chromadb
from chromadb.utils import embedding_functions

multilingual_ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="paraphrase-multilingual-MiniLM-L12-v2"
)

chroma_client = chromadb.Client()
try:
    chroma_client.delete_collection("civic_complaints")
except Exception:
    pass

collection = chroma_client.create_collection(
    name="civic_complaints",
    embedding_function=multilingual_ef
)

# Short, focused search strings (English + Devanagari Hindi + Hinglish) for high-accuracy retrieval
search_texts = [
    "Garbage dumping, overflowing dustbin, uncollected municipal solid waste, street litter. कचरा, कूड़ा, कूड़ेदान खाली नहीं हुआ, गंदगी, सफाई, कचरे का ढेर. kachra, kooda, safai nahi ho rahi, dustbin overflow.",
    "Construction and demolition waste dumping, malba accumulation, rubble, building debris on road. मलबा, निर्माण कचरा, पत्थर, मिट्टी. malba sadak par daal diya, ghar tod kar malba, C&D waste, patthar mitti.",
    "Waste and open garbage burning, smoke, burning dry leaves or plastic in park or street. कचरा जलाना, धुआँ, आग लगाना. kachra jalate hain, kooda jalana, patte jalana, dhuan ghar tak aata hai.",
    "Drinking water supply leakage, burst water pipeline, clean water wastage on road. पानी लीक, पाइपलाइन फट गई, पानी बह रहा है, पानी बर्बाद. pipeline phat gayi, paani leak, nal ka paani beh raha hai.",
    "Sewer overflow, choked drains, blocked naali, dirty sewage water stagnation and bad smell. सीवर ओवरफ्लो, नाली जाम, गटर, गंदा पानी भर गया, बदबू. sewer ka paani, naali jaam, gatar overflow, badbu aa rahi hai.",
    "Illegal tree cutting, felling trees without permission, cutting branches. पेड़ कटाई, पेड़ काटना, बिना अनुमति पेड़ काट रहे हैं. ped kaat rahe hain, bina permission ped katna, illegal tree felling.",
    "Destruction of local biodiversity, heritage sites, sacred groves, wildlife habitat harm. जैव विविधता, धरोहर वृक्ष, वन्यजीव नुकसान. biodiversity damage, sacred grove, heritage trees.",
    "Noise pollution, loud speaker, DJ noise at night, industrial or vehicular air pollution. ध्वनि प्रदूषण, शोर, लाउडस्पीकर, धुआँ, प्रदूषण. raat ko speaker bajte hain, tez awaaz, DJ shor, neend nahi aati, factory smoke.",
    "General civic or environmental grievance escalation when no action is taken on pending complaint. शिकायत पर कोई सुनवाई नहीं हुई, लंबित शिकायत. complaint par koi sunwai nahi hui, no action taken, unresolved complaint escalation."
]

# Full verified facts stored in metadata so URLs/numbers don't dilute vector search
metadatas = [{"full_entry": doc} for doc in documents]

collection.upsert(
    documents=search_texts,
    metadatas=metadatas,
    ids=ids
)

print("ChromaDB updated with focused multilingual search texts and verified metadata.")


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (4 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

ChromaDB updated with focused multilingual search texts and verified metadata.


In [4]:
S = "https://sampark.rajasthan.gov.in"
RSPCB = "https://environment.rajasthan.gov.in/content/environment/en/rajasthan-state-pollution-control-board/ComplaintPage.html"
common = [
    ("(https://rajasthan.gov.in)", f"({S})"),
    ("https://rajasthan.gov.in -> Parivad", f"{S} -> Parivad"),
    ("https://rajasthan.gov.in/ContactUs", f"{S}/ContactUs"),
    ("Rajasthan Municipalities Act, 2009 (Section 226)", "Rajasthan Municipalities Act, 2009"),
    ("Rajasthan Municipalities Act, 2009 (Chapter XI - Drainage and Sewerage)", "Rajasthan Municipalities Act, 2009"),
]
specific = {
    3: [("Source: https://sampark.rajasthan.gov.in", "Source: https://phedwater.rajasthan.gov.in/content/raj/water/public-health-engineering-department/en/citizencorner/phed-control-room-no-.html")],
    6: [("Source: https://sampark.rajasthan.gov.in", "Source: https://environment.rajasthan.gov.in/content/environment/en/rajasthan-state-biodiversity-board/background/Grievance.html")],
    7: [("Source: https://rajasthan.gov.in", f"Source: {RSPCB}")],
}
before = list(documents)
for i in range(len(documents)):
    for old, new in common + specific.get(i, []):
        documents[i] = documents[i].replace(old, new)
print(sum(a != b for a, b in zip(before, documents)), "entries changed")
collection.upsert(documents=search_texts, metadatas=[{"full_entry": d} for d in documents], ids=ids)

7 entries changed


In [5]:
search_texts[5] = (
    "Illegal tree cutting, felling trees without permission, cutting branches in colony or park. "
    "पेड़ कटाई, पेड़ काटना, बिना अनुमति पेड़ काट रहे हैं. "
    "colony ke bahar kuch log bina permission ke purane ped kaat rahe hain, ped katna."
)

search_texts[0] = (
    "Garbage dumping, overflowing dustbin, uncollected municipal solid waste, street litter, garbage piled up in lane causing mosquitoes. "
    "कचरा, कूड़ा, कूड़ेदान खाली नहीं हुआ, गंदगी, सफाई, कचरे का ढेर, मच्छर. "
    "kachra, kooda, kuda pada hai, gali mein kachra, safai nahi ho rahi, dustbin overflow, machhar, mosquitos."
)

search_texts[7] = (
    "Noise pollution, loud speaker, DJ noise at night, late night music disturbing sleep. "
    "ध्वनि प्रदूषण, शोर, लाउडस्पीकर, डीजे, रात को तेज़ आवाज़. "
    "raat ko 12 baje tak speaker bajte hain, neend nahi aati, tez awaaz, DJ shor."
)

collection.upsert(
    documents=search_texts,
    metadatas=[{"full_entry": d} for d in documents],
    ids=ids
)
print("Updated tree cutting, garbage and noise search texts!")


Updated tree cutting, garbage and noise search texts!


In [6]:
def retrieve(complaint):
    """
    Queries the ChromaDB collection with the user's complaint
    and returns the top 3 matching documents.
    """
    # Query the collection for the top 3 closest matches
    results = collection.query(
        query_texts=[complaint],
        n_results=3
    )
    # Extract and return the documents from the nested results structure
    return results['documents'][0]

# Test the retrieve function with the user's query
test_query = "raat ko zor se dj bajta hai"
retrieved_results = retrieve(test_query)

print(f"Query: '{test_query}'\n")
print("Top 3 Matching Documents:")
for idx, doc in enumerate(retrieved_results, 1):
    print(f"{idx}. {doc}")

Query: 'raat ko zor se dj bajta hai'

Top 3 Matching Documents:
1. Noise pollution, loud speaker, DJ noise at night, late night music disturbing sleep. ध्वनि प्रदूषण, शोर, लाउडस्पीकर, डीजे, रात को तेज़ आवाज़. raat ko 12 baje tak speaker bajte hain, neend nahi aati, tez awaaz, DJ shor.
2. Sewer overflow, choked drains, blocked naali, dirty sewage water stagnation and bad smell. सीवर ओवरफ्लो, नाली जाम, गटर, गंदा पानी भर गया, बदबू. sewer ka paani, naali jaam, gatar overflow, badbu aa rahi hai.
3. Garbage dumping, overflowing dustbin, uncollected municipal solid waste, street litter, garbage piled up in lane causing mosquitoes. कचरा, कूड़ा, कूड़ेदान खाली नहीं हुआ, गंदगी, सफाई, कचरे का ढेर, मच्छर. kachra, kooda, kuda pada hai, gali mein kachra, safai nahi ho rahi, dustbin overflow, machhar, mosquitos.


In [7]:
import time
from google.colab import userdata
from google import genai

api_key = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=api_key)

# List primary + fallback models from your Step 1 output
# MODELS = ['gemini-3.5-flash-lite','gemini-3.8-flash','gemini-3.8-pro']
MODELS = ['gemini-3.5-flash-lite','gemini-3.8-flash']

def safe_generate(contents, config=None, retries=3):
    for model_name in MODELS:
        for attempt in range(retries):
            try:
                return client.models.generate_content(
                    model=model_name,
                    contents=contents,
                    config=config
                )
            except Exception as e:
                err_str = str(e)
                if "503" in err_str or "UNAVAILABLE" in err_str or "429" in err_str or "RESOURCE_EXHAUSTED" in err_str:
                    time.sleep(2 ** attempt)  # Wait 1s, 2s, 4s before retrying
                    continue
                raise e  # Raise non-503 errors (like 404 or invalid key) immediately
    raise RuntimeError("All fallback models are temporarily busy. Try again in a moment.")

# Quick test
response = safe_generate('Say "Gemini setup with fallback is ready!"')
print(response.text)


Gemini setup with fallback is ready!


In [8]:
# legacy v2–v3and temperature experiment, not used by the live app

# def draft_complaint(complaint: str, category: str, retrieved_entry: str, temperature: float = 0.2) -> str:
#     """
#     Drafts a structured formal civic complaint letter using ONLY verified facts
#     in retrieved_entry. Falls back to 'not available' for missing fields.
#     """
#     prompt = f"""
# Role: You are 'Hari Awaaz', an official civic complaint assistant for Jaipur, Rajasthan.
# Task: Write a clear, structured formal complaint letter and citizen action summary based ONLY on the 'Retrieved Entry' below.

# STRICT GROUNDING RULES:
# 1. Use ONLY the facts (Authority, Submit via, Rule, Escalate to, Source) present in the 'Retrieved Entry'.
# 2. Do NOT invent, guess, or add any phone numbers, emails, portal links, officer names, or legal Acts that are not explicitly written in 'Retrieved Entry'.
# 3. If any field (such as Rule or Escalate to) states 'not available' in the 'Retrieved Entry', write exactly "not available" for that item.
# 4. Language Matching: If 'User Complaint' is written in Devanagari Hindi script, write the formal "letter" in polite, official Hindi (Devanagari script), keeping all phone numbers, emails, and URLs verbatim. If 'User Complaint' is written in English or Hinglish (Roman script), write the "letter" in clear, formal English.

# REQUIRED OUTPUT FORMAT:
# **To:** [Authority from Retrieved Entry]
# **Subject:** Formal Complaint Regarding {category} in Jaipur

# **Respected Sir/Madam,**
# [2-3 sentences formally describing the citizen's complaint based on 'User Complaint' and requesting prompt inspection and resolution.]

# **Applicable Legal Rule / Act:** [Rule from Retrieved Entry]
# **Where & How to Submit:** [Submit via from Retrieved Entry]
# **Escalation Path (If Unresolved):** [Escalate to from Retrieved Entry]
# **Official Reference Source:** [Source URL from Retrieved Entry]

# User Complaint: {complaint}
# Category: {category}
# Retrieved Entry: {retrieved_entry}
# """

#     try:
#         response = safe_generate(
#             contents=prompt,
#             config=types.GenerateContentConfig(
#                 temperature=temperature
#             )
#         )
#         return response.text.strip()
#     except Exception as e:
#         return f"An error occurred while drafting the complaint: {str(e)}"


In [9]:
# import time

# # Use the real, verified Nagar Nigam Jaipur (JMC) entry from Cell 2
# sample_retrieved = documents[0]
# sample_complaint = "People are throwing plastic waste in the children's park daily and no one is cleaning it."
# sample_category = "Garbage dumping"

# temperatures = [0.0, 0.5, 1.0]

# for temp in temperatures:
#     print(f"=== DRAFT COMPLAINT AT TEMPERATURE {temp} ===")
#     draft = draft_complaint(
#         complaint=sample_complaint,
#         category=sample_category,
#         retrieved_entry=sample_retrieved,
#         temperature=temp
#     )
#     print(draft)
#     print("\n" + "="*40 + "\n")
#     time.sleep(3)  # Pause briefly to stay safely under Free Tier RPM limits


In [10]:
COVERED_CATEGORIES = (
    "Covered categories: garbage dumping, construction & demolition waste, "
    "open burning, water supply leakage, sewer overflow, illegal tree cutting, "
    "biodiversity harm, noise/air pollution, and grievance escalation. "
    "कवर की गई श्रेणियाँ: कचरा, मलबा, खुले में जलाना, पानी रिसाव, सीवर ओवरफ्लो, "
    "पेड़ कटाई, जैव विविधता, ध्वनि/वायु प्रदूषण, शिकायत एस्केलेशन।"
)

def add_categories(result):
    if isinstance(result, dict) and "error" in result:
        result["error"] = str(result["error"]).rstrip(". ") + ". " + COVERED_CATEGORIES
    return result

def build_structured_response(text: str, language: str = "English", location: str = "", timestamp: str = "") -> dict:
    # 1. TRUE RAG RETRIEVAL: Query ChromaDB with raw user text and fetch the top 3 matches
    query_result = collection.query(query_texts=[text], n_results=3)
    metas = query_result["metadatas"][0] if (query_result and query_result.get("metadatas")) else []
    retrieved_entries = [m["full_entry"] for m in metas]

    if not retrieved_entries:
        return {"error": "This issue is not covered under Jaipur civic & environmental complaints."}

    retrieved_context = "\n\n---\n\n".join(
        [f"Option {i+1}: {entry}" for i, entry in enumerate(retrieved_entries)]
    )

    # 2. Set letter language instruction
    lang_clean = language.strip().lower()
    if "hindi" in lang_clean or "हिन्दी" in lang_clean:
        lang_instruction = "Write the 'letter' field completely in polite, official Hindi (Devanagari script), while keeping all phone numbers, emails, and URLs verbatim. Close with [आपका नाम] and [मोबाइल नंबर]."
    else:
        lang_instruction = "Write the 'letter' field in clear, formal English. Close with [Your Name] and [Your Contact Number]."

    location_context = location if location else "Not provided (use placeholder [Exact Landmark / Ward, Jaipur])"

    # 3. Single-call RAG prompt with Date & Time + Location / Landmark header at the top
    prompt = f"""
Role: You are 'Hari Awaaz', a civic complaint drafting tool for Jaipur, Rajasthan (never use 'Hari Awaaz' as the sender's name inside the letter).
Task: Read 'User Complaint' and the 3 'Retrieved Entries' below.
- If the complaint is unrelated to Jaipur civic/environmental issues or fits NONE of the 3 Retrieved Entries, return ONLY:
  {{"error": "This issue is not covered under Jaipur civic & environmental complaints."}}
- Otherwise, pick the single best matching 'Retrieved Entry', extract its verified details, and draft a formal complaint letter on behalf of a concerned Jaipur resident based ONLY on those facts.

STRICT RULES:
1. Use ONLY facts explicitly written in the selected 'Retrieved Entry'. Do NOT guess or invent phone numbers, emails, links, or legal rules.
2. If any field (like rule) is 'not available' in the selected 'Retrieved Entry', write exactly "not available".
3. Language Requirement: {lang_instruction}
4. Timestamp & Location Header Grounding: Start the very top of the letter with two explicit header lines:
   Date & Time: {timestamp}
   Location / Landmark: [insert exact location from 'Location / Landmark', or [Exact Landmark / Ward, Jaipur] if marked Not provided]
5. Return valid JSON with these exact keys:
   - "category": short issue category name (e.g., "Garbage dumping", "Waste/open burning", "Water leakage/wastage", "Tree cutting", "Noise/air pollution", "Sewer overflow/blocked drain", "C&D waste dumping", "Biodiversity damage", "Unresolved grievance escalation")
   - "urgency": "low", "medium", or "high"
   - "authority": exact department/authority name
   - "how_to_submit": exact helpline, app, portal instructions, AND escalation phone numbers/emails from the selected Retrieved Entry
   - "rule": exact Act/Rule or "not available"
   - "source": exact source URL from the selected Retrieved Entry
   - "letter": complete formal complaint letter addressed to the authority (plain text with line breaks, no markdown asterisks)

User Complaint: {text}
Location / Landmark: {location_context}
Report Timestamp: {timestamp}
Retrieved Entries:
{retrieved_context}
"""
    res = safe_generate(
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0.2,
            response_mime_type="application/json"
        )
    )
    parsed = json.loads(res.text.strip())
    return add_categories(parsed)


In [11]:
!pip install -q pycloudflared flask-cors
import json
from datetime import datetime, timezone, timedelta
from flask import Flask, request, jsonify
from flask_cors import CORS
from google.genai import types
import threading, time
from pycloudflared import try_cloudflare

app = Flask(__name__)
CORS(app)

@app.route('/api/complaint', methods=['POST'])
def process_complaint():
    try:
        data = request.get_json(force=True)
        user_text = data.get('complaint', '').strip()
        target_language = data.get('language', 'English').strip()
        location_text = data.get('location', '').strip()

        ist = timezone(timedelta(hours=5, minutes=30))
        default_ts = datetime.now(ist).strftime("%d %b %Y, %I:%M %p IST")
        report_timestamp = data.get('timestamp', '').strip() or default_ts

        if not user_text:
            return jsonify({"error": "Empty complaint text."}), 400
        result = build_structured_response(user_text, target_language, location_text, report_timestamp)
        return jsonify(result)
    except Exception as e:
        return jsonify({"error": str(e)}), 500

# Run Flask on port 5004 and open a fresh Cloudflare Tunnel
threading.Thread(
    target=lambda: app.run(host='0.0.0.0', port=5004, use_reloader=False),
    daemon=True
).start()
time.sleep(2)

tunnel_url = try_cloudflare(port=5004)
print(f"\n✅ NEW LIVE API URL:\n{tunnel_url.tunnel}/api/complaint\n")


 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5004
 * Running on http://172.28.0.12:5004
INFO:werkzeug:Press CTRL+C to quit


Download cloudflared...:   0%|          | 0/40334417 [00:00<?, ?it/s]

 * Running on https://kevin-identity-positioning-talk.trycloudflare.com
 * Traffic stats available on http://127.0.0.1:20241/metrics

✅ NEW LIVE API URL:
https://kevin-identity-positioning-talk.trycloudflare.com/api/complaint



In [12]:
tests = [
    ("Roz subah park ke paas log kachra jalate hain, dhuan ghar tak aata hai.", "open garbage burning"),
    ("Main road par pipeline phat gayi hai, kal se paani beh raha hai.", "Drinking water supply leakage"),
    ("Colony ke bahar kuch log bina permission ke purane ped kaat rahe hain.", "Illegal tree cutting"),
    ("Raat ko 12 baje tak speaker bajte hain, neend nahi aati.", "Noise pollution"),
    ("हमारे मोहल्ले में कूड़ेदान हफ्तों से खाली नहीं हुआ, कचरा सड़क पर फैल रहा है।", "Garbage dumping"),
    ("गली में सीवर का पानी भर गया है और बदबू आ रही है।", "Sewer overflow"),
    ("Padosi ne ghar tod kar sara malba aur patthar sadak par daal diya hai.", "Construction and demolition waste"),
    ("15 din pehle complaint ki thi par abhi tak koi sunwai nahi hui.", "General civic or environmental grievance escalation"),
]

for text, expected in tests:
    res = collection.query(query_texts=[text], n_results=3)
    top3_docs = res["documents"][0]

    # Check if expected match is #1, or within Top 3 (which Gemini sees!)
    in_top1 = expected.lower() in top3_docs[0][:150].lower()
    in_top3 = any(expected.lower() in d[:150].lower() for d in top3_docs)

    status = "PASS (Top 1)" if in_top1 else ("PASS (Top 3)" if in_top3 else "FAIL")
    print(f"{status:<14} | {text[:35]:<35} -> {top3_docs[0][:50]}")



PASS (Top 1)   | Roz subah park ke paas log kachra j -> Waste and open garbage burning, smoke, burning dry
PASS (Top 1)   | Main road par pipeline phat gayi ha -> Drinking water supply leakage, burst water pipelin
PASS (Top 1)   | Colony ke bahar kuch log bina permi -> Illegal tree cutting, felling trees without permis
PASS (Top 1)   | Raat ko 12 baje tak speaker bajte h -> Noise pollution, loud speaker, DJ noise at night, 
PASS (Top 1)   | हमारे मोहल्ले में कूड़ेदान हफ्तों स -> Garbage dumping, overflowing dustbin, uncollected 
PASS (Top 1)   | गली में सीवर का पानी भर गया है और ब -> Sewer overflow, choked drains, blocked naali, dirt
PASS (Top 1)   | Padosi ne ghar tod kar sara malba a -> Construction and demolition waste dumping, malba a
PASS (Top 1)   | 15 din pehle complaint ki thi par a -> General civic or environmental grievance escalatio


In [13]:
import traceback
try:
    r = build_structured_response("book a movie ticket", "English", "Shilp Colony, Jhotwara", "test")
    print(r)
except Exception:
    traceback.print_exc()

{'error': 'This issue is not covered under Jaipur civic & environmental complaints. Covered categories: garbage dumping, construction & demolition waste, open burning, water supply leakage, sewer overflow, illegal tree cutting, biodiversity harm, noise/air pollution, and grievance escalation. कवर की गई श्रेणियाँ: कचरा, मलबा, खुले में जलाना, पानी रिसाव, सीवर ओवरफ्लो, पेड़ कटाई, जैव विविधता, ध्वनि/वायु प्रदूषण, शिकायत एस्केलेशन।'}


In [14]:
import time
hard = [
    "Colony ke bahar kuch log bina permission ke purane ped kaat rahe hain.",
    "Raat ko 12 baje tak speaker bajte hain, neend nahi aati.",
]

for t in hard:
    r = build_structured_response(t, "English", "", "test")
    print(t[:40], "->", r.get("category"), "|", r.get("authority"))
    time.sleep(20)


Colony ke bahar kuch log bina permission -> Tree cutting | Forest Department, Government of Rajasthan
Raat ko 12 baje tak speaker bajte hain,  -> Noise/air pollution | Rajasthan State Pollution Control Board (RSPCB)


In [15]:
import traceback
try:
    print(build_structured_response("Meri gali mein bohot dino se kuda pada hai, mosquitos ho gaye hai bohot saare", "Hindi", "", "test"))
except Exception:
    traceback.print_exc()

def retrieve(complaint):
    """
    Queries the ChromaDB collection with the user's complaint
    and returns the top 3 matching documents.
    """
    # Query the collection for the top 3 closest matches
    results = collection.query(
        query_texts=[complaint],
        n_results=3
    )
    # Extract and return the documents from the nested results structure
    return results['documents'][0]

# Test the retrieve function with the user's query
test_query = "Meri gali mein bohot dino se kuda pada hai, mosquitos ho gaye hai bohot saare"
retrieved_results = retrieve(test_query)

print(f"Query: '{test_query}'\n")
print("Top 3 Matching Documents:")
for idx, doc in enumerate(retrieved_results, 1):
    print(f"{idx}. {doc}")


{'category': 'Garbage dumping', 'urgency': 'medium', 'authority': 'Nagar Nigam Jaipur (Jaipur Municipal Corporation) - Deputy Commissioner (Health) & Executive Engineer (Solid Waste Management)', 'how_to_submit': "JMC Call Centre / Public Grievance Helpline 0141-2747400 (08:00 AM to 10:00 PM, 7 days), 'Jaipur Samadhan' Android mobile app, or online at http://jaipurmc.org (About Us -> Online Services -> Register Your Complaint). Escalate to Commissioner JMC (0141-2740510, commissioner.jmc@rajasthan.gov.in) or JMC Control Room (0141-2742900).", 'rule': 'Solid Waste Management Rules, 2026 & Rajasthan Municipalities Act, 2009', 'source': 'http://jaipurmc.org/Presentation/PublicNotice/FAQs.aspx', 'letter': 'Date & Time: test\nLocation / Landmark: [Exact Landmark / Ward, Jaipur]\n\nसेवा में,\nउप आयुक्त (स्वास्थ्य) एवं अधिशाषी अभियंता (ठोस अपशिष्ट प्रबंधन),\nनगर निगम जयपुर (Jaipur Municipal Corporation)\n\nविषय: गली में कूड़े के ढेर और कचरे के उठान के संबंध में शिकायत।\n\nमहोदय,\n\nसविनय निवे

In [16]:
import traceback
try:
    out = build_structured_response("हमारे मोहल्ले में कूड़ेदान हफ्तों से खाली नहीं हुआ", "Hindi", "Malviya Nagar", "25 Nov 2026")
    print("SUCCESS:", out)
except Exception:
    traceback.print_exc()


SUCCESS: {'category': 'Garbage dumping', 'urgency': 'medium', 'authority': 'Nagar Nigam Jaipur (Jaipur Municipal Corporation) - Deputy Commissioner (Health) & Executive Engineer (Solid Waste Management)', 'how_to_submit': "JMC Call Centre / Public Grievance Helpline 0141-2747400 (08:00 AM to 10:00 PM, 7 days), 'Jaipur Samadhan' Android mobile app, or online at http://jaipurmc.org (About Us -> Online Services -> Register Your Complaint). Escalate to Commissioner JMC (0141-2740510, commissioner.jmc@rajasthan.gov.in) or JMC Control Room (0141-2742900).", 'rule': 'Solid Waste Management Rules, 2026 & Rajasthan Municipalities Act, 2009', 'source': 'http://jaipurmc.org/Presentation/PublicNotice/FAQs.aspx', 'letter': 'Date & Time: 25 Nov 2026\nLocation / Landmark: Malviya Nagar\n\nसेवा में,\nउप आयुक्त (स्वास्थ्य) एवं अधिशाषी अभियंता (ठोस अपशिष्ट प्रबंधन),\nनगर निगम जयपुर (Jaipur Municipal Corporation)\n\nविषय: मोहल्ले में कूड़ेदान हफ्तों से खाली न होने और कचरे के उठान के संबंध में शिकायत।\n\n